In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.getOrCreate()

customers_data = [(1, "John Doe", "Texas"), (2, "Jane Smith", "California")]
customers_schema = StructType(
    [
        StructField("customer_id", IntegerType(), False),
        StructField("full_name", StringType(), False),
        StructField("location", StringType(), False),
    ]
)
customers = spark.createDataFrame(customers_data, schema=customers_schema)

orders_data = [(1001, 1, 101, 5), (1002, 2, 102, 2)]
orders_schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("customer_id", IntegerType(), False),
        StructField("product_id", IntegerType(), False),
        StructField("quantity", IntegerType(), False),
    ]
)
orders = spark.createDataFrame(orders_data, schema=orders_schema)

products_data = [(101, "Carpet,Red"), (102, "Tile,Blue")]
products_schema = StructType(
    [
        StructField("product_id", IntegerType(), False),
        StructField("product_info", StringType(), False),
    ]
)
products = spark.createDataFrame(products_data, schema=products_schema)

In [3]:
customers = customers.withColumn(
    "split_name",
    split(customers["full_name"], " "),
)
customers = customers.withColumn(
    "first_name",
    customers["split_name"].getItem(0),
)
customers = customers.withColumn(
    "last_name",
    customers["split_name"].getItem(1),
)
customers = customers.drop("split_name", "full_name")

# Split the product_info column into product_type and product_color
products = products.withColumn(
    "split_info",
    split(products["product_info"], ","),
)
products = products.withColumn(
    "product_type",
    products["split_info"].getItem(0),
)
products = products.withColumn(
    "product_color",
    products["split_info"].getItem(1),
)
products = products.drop("split_info", "product_info")

# Join all three dataframes
df = orders.join(customers, "customer_id").join(products, "product_id")